In [57]:
import json, pandas as pd, numpy as np, statistics, re
from pathlib import Path
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

BASE = Path("/Users/matteogevi/Aurora-History-MVP/data/out")
CHUNKS_PATH = BASE / "chunks.paragraphs.jsonl"
TOC_PATH = BASE / "toc.json"

rows = [json.loads(l) for l in open(CHUNKS_PATH, "r", encoding="utf-8") if l.strip()]
print(f"✅ Loaded {len(rows)} chunks")

df = pd.DataFrame(rows)
print("Columns:", list(df.columns))

print(
    f"rows: {len(df)} | "
    f"unique sections: {df['section_id'].nunique()} | "
    f"avg chars: {df['text'].str.len().mean():.1f} | "
    f"median: {df['text'].str.len().median():.0f} | "
    f">2000 chars: {(df['text'].str.len()>2000).sum()} | "
    f"empty: {(df['text'].str.len()==0).sum()}"
)

def preview_row(r):
    t = (r.get("text") or "").replace("\n", " ")
    return {
        "chunk_id": r["chunk_id"],
        "section_id": r["section_id"],
        "section_title": r.get("section_title"),
        "level": r.get("level"),
        "page_range": r.get("page_range"),
        "len": len(t),
        "text_preview": t[:200],
    }

preview_df = pd.DataFrame([preview_row(r) for r in rows])
preview_df.head(10)

print("total chunks:", len(df))
print("unique section_id:", df["section_id"].nunique())
print("missing section_id:", df["section_id"].isna().sum())
print("unique texts:", df["text"].nunique())

# Check if texts are empty or repeated
print("\nEmpty texts:", (df["text"].str.strip().str.len() == 0).sum())
print(df["text"].value_counts().head(3))

✅ Loaded 2405 chunks
Columns: ['chunk_id', 'chunk_seq', 'doc_key', 'section_id', 'section_title', 'level', 'page_range', 'text']
rows: 2405 | unique sections: 180 | avg chars: 906.8 | median: 935 | >2000 chars: 3 | empty: 0
total chunks: 2405
unique section_id: 180
missing section_id: 0
unique texts: 1841

Empty texts: 0
text
The success of ChatGPT prompted a wave of text-based conversational bots.\n\nHow‐\never, text isn’t the only interface for conversational agents.\n\nVoice assistants such as\nGoogle Assistant, Siri, and Alexa have been around for years.15 3D conversational\nbots are already common in games and gaining traction in retail and marketing.\n\n\nOne use case of AI-powered 3D characters is smart NPCs, non-player characters (see\nNVIDIA’s demos of Inworld and Convai).16 NPCs are essential for advancing the\nstoryline of many games.\n\nWithout AI, NPCs are typically scripted to do simple\nactions with a limited range of dialogues.\n\nAI can make these NPCs much smarter.\n\

In [35]:
'''Chunking Display'''

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    raw = [ln for ln in f if ln.strip()][:3]
print("first 3 lines (raw):")
for ln in raw:
    s = ln[:200].replace("\n"," ")
    print(s + ("..." if len(ln) > 200 else ""))

# 2) load rows
rows = [json.loads(ln) for ln in open(CHUNKS_PATH, "r", encoding="utf-8") if ln.strip()]
print("rows:", len(rows))
print("keys example:", sorted(rows[0].keys()))

# 3) score string-like fields to choose best text key (prefer 'text' if present)
lens = defaultdict(list)
for r in rows:
    for k, v in r.items():
        if isinstance(v, str):
            lens[k].append(len(v.strip()))

def score(arr):
    return (sum(1 for x in arr if x > 0),
            round(statistics.mean(arr), 1) if arr else 0)

candidates = {k: score(arr) for k, arr in lens.items()}
print("string fields (nonempty_count, avg_len):")
for k, (n, avg) in sorted(candidates.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True):
    print(f"- {k}: {n}, {avg}")

TEXT_KEY = "text" if "text" in rows[0] else \
           max(candidates, key=lambda k: (candidates[k][0], candidates[k][1])) if candidates else None
print("Chosen TEXT_KEY:", TEXT_KEY)

# 4) small embedding demo with all-MiniLM-L6-v2
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

N = min(8, len(rows))
examples = []
for i, r in enumerate(rows[:N]):
    txt = r.get(TEXT_KEY, "") or ""
    emb = model.encode([txt], normalize_embeddings=True)[0]
    examples.append({
        "chunk_id": r.get("chunk_id") or f"chunk-{i}",
        "section_id": r.get("section_id"),
        "section_title": r.get("section_title"),
        "level": r.get("level"),
        "page_range": r.get("page_range"),
        "len": len(txt),
        "content_preview": txt.replace("\n"," ")[:220],
        "embedding_dim": len(emb),
        "embedding_first5": np.round(emb[:5], 4).tolist(),
    })

pd.DataFrame(examples)

first 3 lines (raw):
{"chunk_id": "b203c98fadb3f0fe::h1-1__cover::c000001", "chunk_seq": 1, "doc_key": "b203c98fadb3f0fe", "section_id": "h1-1__cover", "section_title": "Cover", "level": 1, "page_range": [1, 5], "text": "...
{"chunk_id": "b203c98fadb3f0fe::h1-1__cover::c000002", "chunk_seq": 2, "doc_key": "b203c98fadb3f0fe", "section_id": "h1-1__cover", "section_title": "Cover", "level": 1, "page_range": [1, 5], "text": "...
{"chunk_id": "b203c98fadb3f0fe::h1-1__cover::c000003", "chunk_seq": 3, "doc_key": "b203c98fadb3f0fe", "section_id": "h1-1__cover", "section_title": "Cover", "level": 1, "page_range": [1, 5], "text": "...
rows: 2405
keys example: ['chunk_id', 'chunk_seq', 'doc_key', 'level', 'page_range', 'section_id', 'section_title', 'text']
string fields (nonempty_count, avg_len):
- text: 2405, 906.8
- chunk_id: 2405, 58.0
- section_id: 2405, 31.0
- section_title: 2405, 21.7
- doc_key: 2405, 16
Chosen TEXT_KEY: text


,chunk_id,section_id,section_title,level,page_range,len,content_preview,embedding_dim,embedding_first5
0,b203c98fadb3f0fe::h1-1__cover::c000001,h1-1__cover,Cover,1,"[1, 5]",877,Chip Huyen AI Engineering Building Applicatio...,384,"[-0.04230000078678131, -0.06719999760389328, 0..."
1,b203c98fadb3f0fe::h1-1__cover::c000002,h1-1__cover,Cover,1,"[1, 5]",984,The book also introduces a practical framework...,384,"[-0.03460000082850456, -0.06340000033378601, 0..."
2,b203c98fadb3f0fe::h1-1__cover::c000003,h1-1__cover,Cover,1,"[1, 5]",956,She is a remarkable teacher and writer whose w...,384,"[-0.08209999650716782, -0.05660000070929527, 0..."
3,b203c98fadb3f0fe::h1-1__cover::c000004,h1-1__cover,Cover,1,"[1, 5]",957,"—Vittorio Cretella, former global CIO, P&G and...",384,"[-0.10559999942779541, -0.03799999877810478, 0..."
4,b203c98fadb3f0fe::h1-1__cover::c000005,h1-1__cover,Cover,1,"[1, 5]",937,Unlike other books that focus on tools or curr...,384,"[-0.03519999980926514, -0.05900000035762787, 0..."
5,b203c98fadb3f0fe::h1-1__cover::c000006,h1-1__cover,Cover,1,"[1, 5]",769,This book is an essential resource for anyone ...,384,"[-0.054999999701976776, -0.020899999886751175,..."
6,b203c98fadb3f0fe::h1-2__copyright::c000001,h1-2__copyright,Copyright,1,"[6, 6]",597,978-1-098-16630-4 [LSI] AI Engineering by Chip...,384,"[-0.06589999794960022, -0.026499999687075615, ..."
7,b203c98fadb3f0fe::h1-2__copyright::c000002,h1-2__copyright,Copyright,1,"[6, 6]",751,Acquisitions Editor: Nicole Butterfield Indexe...,384,"[-0.06970000267028809, 0.0414000004529953, -0...."


In [61]:
'''Chunk length & duplicate sniff'''

rows = [json.loads(l) for l in open(CHUNKS_PATH, "r", encoding="utf-8") if l.strip()]
print("rows loaded:", len(rows))

# ---- Auto-detect TEXT_KEY (favor 'text' if present) ----
lens = defaultdict(list)
for r in rows:
    for k, v in r.items():
        if isinstance(v, str):
            lens[k].append(len(v.strip()))

def score(arr):
    nonempty = sum(1 for x in arr if x > 0)
    avg = statistics.mean(arr) if arr else 0.0
    return (nonempty, avg)

candidates = {k: score(arr) for k, arr in lens.items()}

if "text" in candidates:
    TEXT_KEY = "text"  # your new schema; be explicit if available
else:
    TEXT_KEY = max(candidates, key=lambda k: (candidates[k][0], candidates[k][1])) if candidates else None

print("Using TEXT_KEY:", TEXT_KEY, "->", candidates.get(TEXT_KEY))

# ---- Chunk length & duplicate sniff ----
def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

if TEXT_KEY:
    lengths = [len((r.get(TEXT_KEY) or "").strip()) for r in rows]
    print(
        "n:", len(rows),
        "| avg chars:", round(statistics.mean(lengths), 1) if lengths else 0,
        "| median:", int(statistics.median(lengths)) if lengths else 0,
        "| >2000 chars:", sum(int(L > 2000) for L in lengths),
        "| empty:", sum(int(L == 0) for L in lengths),
    )

    seen = set()
    dups = 0
    for r in rows:
        key = normalize(r.get(TEXT_KEY) or "")[:400]  # prefix to keep memory low
        if key in seen:
            dups += 1
        seen.add(key)
    print("potential duplicates:", dups)
else:
    print("No textual key detected; rows may be malformed.")

# ---- Nice DataFrame preview for your schema ----
def preview_row(r):
    txt = (r.get(TEXT_KEY) or "").replace("\n", " ")
    return {
        "chunk_id": r.get("chunk_id"),
        "section_id": r.get("section_id"),
        "section_title": r.get("section_title"),
        "level": r.get("level"),
        "page_range": r.get("page_range"),
        "len": len(txt),
        "text_preview": txt[:220],
    }

df = pd.DataFrame([preview_row(r) for r in rows])
display(df.head(12))

# ---- Quick per-section summary (helps spot skew) ----
sec_summary = (
    df.groupby(["level", "section_id", "section_title"], dropna=False)["len"]
      .agg(n_chunks="count", avg_len="mean", median_len="median")
      .reset_index()
      .sort_values(["level", "n_chunks"], ascending=[True, False])
)
display(sec_summary.head(15))

rows loaded: 2405
Using TEXT_KEY: text -> (2405, 906.8029106029106)
n: 2405 | avg chars: 906.8 | median: 935 | >2000 chars: 3 | empty: 0
potential duplicates: 593


,chunk_id,section_id,section_title,level,page_range,len,text_preview
0,b203c98fadb3f0fe::h1-1__cover::c000001,h1-1__cover,Cover,1,"[1, 5]",877,Chip Huyen AI Engineering Building Applicatio...
1,b203c98fadb3f0fe::h1-1__cover::c000002,h1-1__cover,Cover,1,"[1, 5]",984,The book also introduces a practical framework...
2,b203c98fadb3f0fe::h1-1__cover::c000003,h1-1__cover,Cover,1,"[1, 5]",956,She is a remarkable teacher and writer whose w...
3,b203c98fadb3f0fe::h1-1__cover::c000004,h1-1__cover,Cover,1,"[1, 5]",957,"—Vittorio Cretella, former global CIO, P&G and..."
4,b203c98fadb3f0fe::h1-1__cover::c000005,h1-1__cover,Cover,1,"[1, 5]",937,Unlike other books that focus on tools or curr...
5,b203c98fadb3f0fe::h1-1__cover::c000006,h1-1__cover,Cover,1,"[1, 5]",769,This book is an essential resource for anyone ...
6,b203c98fadb3f0fe::h1-2__copyright::c000001,h1-2__copyright,Copyright,1,"[6, 6]",597,978-1-098-16630-4 [LSI] AI Engineering by Chip...
7,b203c98fadb3f0fe::h1-2__copyright::c000002,h1-2__copyright,Copyright,1,"[6, 6]",751,Acquisitions Editor: Nicole Butterfield Indexe...
8,b203c98fadb3f0fe::h1-2__copyright::c000003,h1-2__copyright,Copyright,1,"[6, 6]",782,The views expressed in this work are those of ...
9,b203c98fadb3f0fe::h1-3__table-of-contents::c00...,h1-3__table-of-contents,Table of Contents,1,"[7, 12]",275,Table of Contents Preface. . . . . . . . . . ....


,level,section_id,section_title,n_chunks,avg_len,median_len
9,1,h1-3__table-of-contents,Table of Contents,23,875.130435,921.0
1,1,h1-12__chapter-8-dataset-engineering,Chapter 8. Dataset Engineering,6,861.166667,892.5
7,1,h1-1__cover,Cover,6,913.333333,946.5
4,1,h1-16__index,Index,5,5790.000000,1218.0
6,1,h1-18__colophon,Colophon,4,763.000000,900.0
0,1,h1-11__chapter-7-finetuning,Chapter 7. Finetuning,3,694.000000,860.0
5,1,h1-17__about-the-author,About the Author,3,896.666667,898.0
8,1,h1-2__copyright,Copyright,3,710.000000,751.0
10,1,h1-4__preface,Preface,3,683.666667,815.0
12,1,h1-6__chapter-2-understanding-foundation-models,Chapter 2. Understanding Foundation Models,3,700.666667,831.0


In [49]:
'''Embedding norms + self-nearest-neighbor sanity'''

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [r.get("content","") for r in rows[:500]]  # subset for speed
X = model.encode(texts, normalize_embeddings=True).astype("float32")

# Check unit norms
norms = np.linalg.norm(X, axis=1)
print("norm mean:", float(norms.mean()), "min:", float(norms.min()), "max:", float(norms.max()))

# Self NN (exclude self)
S = X @ X.T
np.fill_diagonal(S, -1.0)
top_idx = S.argmax(axis=1)
sample = [(i, int(top_idx[i]), float(S[i, top_idx[i]])) for i in range(min(10, len(texts)))]
print("top neighbors (i -> j, cos):", sample[:5])

norm mean: 0.9999998807907104 min: 0.9999999403953552 max: 0.9999999403953552
top neighbors (i -> j, cos): [(0, 1, 1.0), (1, 0, 1.0), (2, 0, 1.0), (3, 0, 1.0), (4, 0, 1.0)]


In [67]:
'''“Same section” vs random similarity'''

# Use correct key from your JSONL
sec_ids = [r.get("section_id") for r in rows[:500]]
texts = [r.get("text") for r in rows[:500]]

# Generate embeddings
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X = model.encode(texts, normalize_embeddings=True).astype("float32")

pairs_same, sims_same = 0, []
pairs_diff, sims_diff = 0, []

for _ in range(200):  # more samples = smoother stats
    i, j = random.sample(range(len(sec_ids)), 2)
    sim = float(X[i] @ X[j])
    if sec_ids[i] and sec_ids[j] and sec_ids[i] == sec_ids[j]:
        pairs_same += 1
        sims_same.append(sim)
    else:
        pairs_diff += 1
        sims_diff.append(sim)

def mean_safe(a): return round(sum(a) / len(a), 4) if a else None
print(f"same-sec avg cos: {mean_safe(sims_same)} (n={pairs_same})")
print(f"diff-sec avg cos: {mean_safe(sims_diff)} (n={pairs_diff})")

# ---------------- SUMMARY INSIGHT ----------------
if sims_same and sims_diff:
    gap = mean_safe(sims_same) - mean_safe(sims_diff)
    print(f"Δ similarity (same - diff): {gap:.4f}")
    if gap > 0.10:
        print("✅ clear semantic separation between sections")
    elif gap > 0.05:
        print("⚠️ moderate separation — good, might improve with tuned chunking or larger model")
    else:
        print("❌ low separation — embeddings might be too generic or overlapping heavily")
else:
    print("⚠️ Not enough same-section pairs to compare meaningfully.")

same-sec avg cos: 0.441 (n=13)
diff-sec avg cos: 0.2859 (n=187)
Δ similarity (same - diff): 0.1551
✅ clear semantic separation between sections


In [39]:
'''Duplicated Chunks'''

PATH = "/Users/matteogevi/Aurora-History-MVP/data/out/chunks.paragraphs.jsonl"
rows = [json.loads(l) for l in open(PATH, "r", encoding="utf-8") if l.strip()]

def norm(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

# exact full-text duplicate rate
seen = {}
dups_exact = 0
for r in rows:
    k = hashlib.sha1(norm(r["text"]).encode("utf-8")).hexdigest()
    dups_exact += k in seen
    seen[k] = True

print("exact duplicates (full-text normalized):", dups_exact)

exact duplicates (full-text normalized): 564
